# STAGE: Spatio-Temporal Allelic Geostatistic Estimation
**Part 1:Variogram Fitting**

Geostatistics was originally designed for continuous numerical data (like temperature or elevation). The STAGE framework bridges this gap, allowing us to perform spatiotemporal geostatistical analysis on **categorical genetic data** (e.g., *Avr-Pik* variants).

This notebook run the STAGE framework using a simulated data. We will:
1. Calculate empirical spatio variograms using a simulated *Avr-Pik* sample dataset.
2. Fit a theoretical Spherical model to extract spatial parameters (Range, Sill, Nugget).

## 1. Setup the Environment
Instructions for Reviewers: This notebook is fully automated to run in a temporary Colab environment. The first code cell will configure the workspace, install necessary geostatistical libraries (such as scikit-gstat), and automatically download the required datasets (sage_spatiotemporal_sample_data.csv; avrpik_gt_distmatdf.csv).

Simply select Runtime > Run all from the top menu to execute the entire pipeline.

In [1]:
!pip install numpy pandas scipy scikit-learn scikit-gstat matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 710.3/710.3 kB 5.3 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import math

from scipy.spatial import distance_matrix

from scipy.optimize import curve_fit
from skgstat.models import spherical
from sklearn.metrics import r2_score, root_mean_squared_error

import matplotlib.pyplot as plt
import seaborn as sns

# Download the sample data to run SAGE in the Colab environment
!gdown "1bJFR6TXhEgeFPXj-dT_r8tZL_KIOi16K" -O stage_spatiotemporal_sample_data.csv

# Download the genetic distance matrix file
!gdown "1Vx8vU1Jb2vrLcPA0ahk9jEfhrqMA8837" -O avrpik_gt_distmatdf.csv

## 2. Calculate Empirical Variograms
To test our framework, we used a simulated data set with 100 sampling locations across a landscape and 3 years (2015-2017).

The functions below calculate the pairwise geographic distance and genetic variance between every possible pair of overall samples. We then "bin" these distances into specific spatial lags (e.g., grouping all farm-pairs that are 0-5km apart, 5-10km apart, etc.) to calculate the average genetic variance at that distance.

In [ ]:
def genevariance(gt_coords_df):
    "calculate genetic variance"
    n_samples = len(gt_coords_df)
    genetic_dists = []
    geo_dists = []
    for i in range(n_samples):
        gti = gt_coords_df.at[i,'Genotype']
        xcoordi = gt_coords_df.at[i,'CoordX']
        ycoordi = gt_coords_df.at[i,'CoordY']

        for j in range(n_samples):
            if (j != i):
                gtj = gt_coords_df.at[j,'Genotype']
                xcoordj = gt_coords_df.at[j,'CoordX']
                ycoordj = gt_coords_df.at[j,'CoordY']

                "calculate genetic variance"
                gene_dist = gt_distmat.at[gti, gtj]
                gene_varince = (gene_dist ** 2) * 0.5  # semi-variance

                "calculate samples geographic distance"
                geo_dist = math.dist([xcoordi, ycoordi],[xcoordj, ycoordj])

                genetic_dists.append(gene_varince)
                geo_dists.append(geo_dist)

    "save to semivarice to a dataframe"
    vario_df = pd.DataFrame({"geo_dist":geo_dists, "genetic_dist":genetic_dists})

    return vario_df


def binningaverage(vario_df, lag):
    "binning and group into lags"
    vario_df["lag"] = vario_df["geo_dist"].apply(lambda x: math.ceil((x - 0.5 * lag) / lag)) # the first bin is [0, 0.5*lag]
    vario_df = vario_df.sort_values(by=["geo_dist"])
    vario_gpdf = vario_df.groupby(['lag']).agg(genetic_dist_mean=("genetic_dist", "mean"),
                                               geo_dist_mean=("geo_dist", "mean"),
                                               pair_count=("genetic_dist", "count")).reset_index()

    "create a new column to save the genetic variance"
    gencol = "genetic_dist_" + str(lag)
    vario_gpdf[gencol] = vario_gpdf["genetic_dist_mean"]

    return vario_gpdf


"import the genetic distance matrix"
gt_distmat = pd.read_csv("avrpik_gt_distmatdf.csv", index_col=0)

"import genotype sample data"
gt_coords_df = pd.read_csv("stage_spatiotemporal_sample_data.csv")

"calculate genetic variogram"
vario_df = genevariance(gt_coords_df)

"explore lag distances"
lags = [5, 10, 15, 20, 25, 30]

fig, axs = plt.subplots(nrows=3, ncols=2, figsize=(10, 8))
for i in range(len(lags)):
    lag = lags[i]
    row = i // 2
    col = i % 2

    "binning and average of variance"
    vario_gpdf = binningaverage(vario_df, lag)

    "plot genetic variance to distance"
    gencol = "genetic_dist_" + str(lag)
    sns.scatterplot(x=vario_gpdf['geo_dist_mean'], y=vario_gpdf[gencol], ax=axs[row][col])
    axs[row][col].set_xlim(-5, 150)
    axs[row][col].set_ylim(-1, 35)

    "save these variogram files"
    vario_df.to_csv("vario_df_" + str(lag) + ".csv")
    vario_gpdf.to_csv("vario_group_df_" + str(lag) + ".csv")

fig.tight_layout()
fig.savefig("varigrams.png", dpi=300)

## 3. Fit the Theoretical Spherical Model

An empirical variogram shows us the raw data points, but to extract meaningful epidemiological metrics, we must fit a theoretical model to these points. We use a **Spherical Model**, wrapped using `scipy.optimize.curve_fit`.

This optimization allows us to extract three critical parameters of the disease spread:
* **Range ($a$):** The distance at which genetic correlation stops. Beyond this distance, pathogen populations are genetically independent.
* **Partial Sill ($b$):** The maximum genetic variance (the plateau of the curve).
* **Nugget ($c$):** The y-intercept, representing micro-scale genetic variation or sampling error at a distance of zero.

Below, we iterate through multiple lag distances (5, 10, 15, 20, 25, 30 km) to evaluate model performance across different spatial granularities, recording the RMSE, R-squared, and AIC for each fit.

In [ ]:
def f(x, a, b, c):
    """
    As the spherical function is compiled using numba, we wrap the function
    in order to let curve_fit correctly infer the parameters.

    In default, the parameter nugget (here is c) is 0

    """

    return spherical(x, a, b, c)


year = 201520162017
lags = [5, 10, 15, 20, 25, 30]

sns.set_style('darkgrid')
fig, axs = plt.subplots(nrows=3, ncols=2, figsize=(10, 8))

"explore model fitting by lag"
lag_lst = []
p00_lst = [] # initial value for the effective range
p01_lst = [] # initial value for the partial sill of the variogram
p02_lst = [] # initial value for the nugget of the variogram
cof0_lst = [] #estimated value the effective range
cof1_lst = [] #estimated value for the partial sill of the variogram
cof2_lst = [] #estimated value for the nugget of the variogram
cov0_lst = [] #standard error of the effective range
cov1_lst = [] #standard error of the partial sill
cov2_lst = [] #standard error of the nugget
rmse_lst = [] # rmse
r2_lst = []
aics = []
num_lags_lst = []
infodicts = []


for i in range(len(lags)):
    lag = lags[i]
    row = i // 2
    col = i % 2

    gencol = "genetic_dist_" + str(lag)
    vario_gpdf = pd.read_csv("vario_group_df_" + str(lag) + ".csv", index_col=0)

    x = vario_gpdf[vario_gpdf['geo_dist_mean'] <= 150]['geo_dist_mean'].to_list() #half of the max distance
    y = vario_gpdf[vario_gpdf['geo_dist_mean'] <= 150][gencol].to_list()
    xi = np.linspace(0, 150, 100)

    "constrained fit"
    init_vals = [30, 1, 0]
    cof, cov, info0, info1, info2 = curve_fit(f, x, y, p0=init_vals, full_output=True, bounds=(0, (np.max(x), np.max(y), np.max(y))))
    yi = list(map(lambda x: f(x, *cof), xi))

    "calculate standard error of estimated parameters"
    perr = np.sqrt(np.diag(cov)) # standard errors of estimated parameters

    "calculate RMSE"
    rmse_c = root_mean_squared_error(y, f(x, *cof))

    "calculate R squared"
    r2 = r2_score(y, f(x, *cof))

    "calculate AIC"
    residuals = y - f(x, *cof)
    rss = np.sum(residuals**2) # residual sum of squares
    aic = 2 * 3 + len(y) * np.log(rss / len(y))

    sns.scatterplot(x=x, y=y, markers='p', color='blue', label='experimental', ax=axs[row][col])
    sns.lineplot(x=xi, y=yi, markers='-', color='k',label=str(lag), ax=axs[row][col])
    axs[row][col].set_ylim(-0, 35)

    "save parameters"
    lag_lst.append(lag)
    p00_lst.append(init_vals[0])
    p01_lst.append(init_vals[1])
    p02_lst.append(init_vals[2])
    cof0_lst.append(cof[0])
    cof1_lst.append(cof[1])
    cof2_lst.append(cof[2])
    cov0_lst.append(perr[0])
    cov1_lst.append(perr[1])
    cov2_lst.append(perr[2])

    rmse_lst.append(rmse_c)
    r2_lst.append(r2)
    aics.append(aic)
    num_lags_lst.append(len(x))
    infodicts.append([info0, info1, info2])

"save model fitting parameters and performance to a dataframe"
fitted_variogram = pd.DataFrame(data={'lag':lag_lst, 'range_init':p00_lst,
                                      'partial_sill_init':p01_lst,
                                      'nugget_init':p02_lst,
                                      'range':cof0_lst, 'partial_sill':cof1_lst,
                                      'nugget':cof2_lst, 'range_std':cov0_lst,
                                      'partial_sill_std':cov1_lst, 'nugget_std':cov2_lst,
                                      'rmse':rmse_lst, 'r2':r2_lst, 'aic':aics,
                                      'num_lags':num_lags_lst,
                                      'infordict':infodicts})
"save figure and data"
fitted_variogram.to_csv("fitted_variogram_model_" + str(year) + ".csv")
fig.suptitle(str(year))
fig.tight_layout()
fig.savefig("fitting_experimental_varigrams_" + str(year) + ".png", dpi=300)

In [ ]:
print("**Congratulations! You successfully finished the running!**")